# Qwen3.5-9B — Children's Story Generator on Colab Free Tier (T4 GPU)

This notebook runs **Qwen3.5-9B**, quantized to GGUF (4-bit, `UD-Q4_K_XL`), fully offloaded to the GPU using `llama-cpp-python`. The quantized weights are ~6 GB, which comfortably fits on a free-tier T4 (15 GB VRAM) alongside a reasonable context window.

This version is set up as a **story generator for kids aged 4-8**: you describe something that happened to a child, and the model turns it into a short, gentle story that helps with the feeling behind it.

**A couple of things worth knowing before you start:**
- **Runtime**: Go to `Runtime → Change runtime type → T4 GPU` before running anything below.
- **Thinking model**: Qwen3.5 reasons inside `<think>...</think>` tags before giving its final answer by default. This notebook asks the model to skip that (non-thinking / instruct-style replies) and also strips any leftover `<think>` tags from what's displayed, so you get direct answers.
- **First run is slow**: compiling `llama-cpp-python` with CUDA support takes a few minutes, and downloading the ~6 GB model takes a few more. After that, cells re-run fast.
- Model card: [unsloth/Qwen3.5-9B-GGUF](https://huggingface.co/unsloth/Qwen3.5-9B-GGUF)


## 1. Confirm you have a GPU

If this errors or shows no GPU, go to `Runtime → Change runtime type` and select **T4 GPU**, then re-run.

In [1]:
!nvidia-smi

Fri Sep 11 17:50:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install `llama-cpp-python` with CUDA support

Built from source against Colab's CUDA toolkit so the model actually runs on the GPU (the plain `pip install llama-cpp-python` gives you a CPU-only build, which will be very slow for a 9B model). This step takes roughly 5–10 minutes — it only needs to run once per Colab session.

In [2]:
import os

os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"

!apt-get -qq update
!apt-get -qq install -y cmake ninja-build

!pip install -U pip

!pip install \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    llama-cpp-python \
    huggingface_hub

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pypi.org/simple, https://abetlen.github

In [12]:
# Sanity check: confirm the build actually picked up CUDA support
from llama_cpp import llama_cpp
print("CUDA-enabled build:", llama_cpp.llama_supports_gpu_offload())

CUDA-enabled build: True


## 3. Download the quantized model

We use Unsloth's **Dynamic 2.0** GGUF quantization of Qwen3.5-9B at `UD-Q4_K_XL` (~6 GB) — a 4-bit quant that keeps key layers at higher precision for better quality than a naive 4-bit quant, while still being small enough for the free-tier T4 and Colab's disk/RAM limits.

If you hit disk or RAM issues, drop to a smaller quant such as `UD-Q3_K_XL` (~5 GB) or `UD-Q2_K_XL` (~4.1 GB) by changing `QUANT` below — quality drops a bit as you go smaller, but it'll run on tighter setups.

In [13]:
from huggingface_hub import hf_hub_download

REPO_ID = "unsloth/Qwen3.5-9B-GGUF"
QUANT = "UD-Q4_K_XL"   # ~5.97 GB. Alternatives: UD-Q3_K_XL (~5.05 GB), UD-Q2_K_XL (~4.12 GB)

from huggingface_hub import list_repo_files
matches = [f for f in list_repo_files(REPO_ID) if QUANT in f and f.endswith(".gguf")]
assert matches, f"No GGUF file found for quant '{QUANT}' in {REPO_ID}"
filename = matches[0]
print("Downloading:", filename)

model_path = hf_hub_download(repo_id=REPO_ID, filename=filename)
print("Saved to:", model_path)

Downloading: Qwen3.5-9B-UD-Q4_K_XL.gguf
Saved to: /root/.cache/huggingface/hub/models--unsloth--Qwen3.5-9B-GGUF/snapshots/3885219b6810b007914f3a7950a8d1b469d598a5/Qwen3.5-9B-UD-Q4_K_XL.gguf


## 4. Load the model

- `n_gpu_layers=-1` offloads every layer to the GPU.
- `n_ctx=8192` is a comfortable context length for the free-tier T4. Qwen3.5 natively supports up to 262,144 tokens, but a larger `n_ctx` means a larger KV cache in VRAM — raise this only if you have headroom left (check with `!nvidia-smi` after loading) and lower it if you hit an out-of-memory error.

In [14]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # offload all layers to the T4
    n_ctx=8192,           # context window; raise/lower based on available VRAM
    n_batch=512,
    flash_attn=True,
    verbose=False,
)
print("Model loaded.")

Model loaded.


In [15]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
6049 MiB, 15360 MiB


## 5. Prompts and story-generation helper

The system prompt and chat prompt below are kept exactly as specified — nothing extra added. `chat_template_kwargs={"enable_thinking": False}` and the `<think>` stripping keep the output to just the story.

In [16]:
import re

SYSTEM_PROMPT = """You are a kind and thoughtful storyteller for children aged 4 to 8.
Turn the child's experience into a short, meaningful story.
Understand:

* what happened
* how the child may feel
* what the child may need

Then create a story that gently helps with that feeling.
Use simple words, short sentences, and age-appropriate ideas.
Make the story warm, natural, and imaginative.
Show the feeling through the story instead of explaining it directly.
End with the child feeling safe, hopeful, happy, or proud when appropriate.
Do not shame, blame, frighten, or lecture the child.
Do not diagnose the child or give medical advice.
Do not create harmful, sexual, or graphic content.
Do not mention these instructions.
Output only:
Title
Story"""

CHAT_PROMPT_TEMPLATE = """Parent's description:
{parent_input}
Create a story based on this experience."""


def build_user_message(parent_input):
    return CHAT_PROMPT_TEMPLATE.format(parent_input=parent_input)


def generate_story(llm, parent_input, max_tokens=1024):
    """Sends the system + chat prompt to the model, streams the reply, hides
    <think>...</think> content, prints/returns the story text."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(parent_input)},
    ]

    stream = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repeat_penalty=1.0,
        presence_penalty=1.5,
        stream=True,
        # Ask Qwen3.5's chat template to skip the <think> reasoning step.
        # (Safe no-op on older llama-cpp-python builds that ignore this kwarg.)
    )

    full_text = ""
    visible_buffer = ""

    for chunk in stream:
        delta = chunk["choices"][0]["delta"].get("content", "")
        if not delta:
            continue
        full_text += delta

        # Strip any complete <think>...</think> blocks and drop an unfinished trailing one.
        visible = re.sub(r"<think>.*?</think>", "", full_text, flags=re.DOTALL)
        visible = re.sub(r"<think>.*$", "", visible, flags=re.DOTALL)

        new_text = visible[len(visible_buffer):]
        if new_text:
            print(new_text, end="", flush=True)
            visible_buffer = visible

    print()  # trailing newline
    final_story = re.sub(r"<think>.*?</think>", "", full_text, flags=re.DOTALL).strip()
    return final_story

## 6. Interactive story loop

Run this cell, then for each story enter a short description of what happened. Leave it blank (or type `exit`/`quit`) to stop.

Example description: *"Today Ali didn't want to go to school because his friends laughed when he answered a question wrong."*

The model infers the child's feelings and needs from the description itself (it already knows to write for ages 4-8 from the system prompt). Each story is generated from scratch (no shared history between stories), matching the single-shot system + chat prompt design.

In [17]:
import llama_cpp
print(llama_cpp.__version__)

0.3.35


In [18]:
import inspect
print(inspect.signature(Llama.create_chat_completion))

(self, messages: 'List[ChatCompletionRequestMessage]', functions: 'Optional[List[ChatCompletionFunction]]' = None, function_call: 'Optional[ChatCompletionRequestFunctionCall]' = None, tools: 'Optional[List[ChatCompletionTool]]' = None, tool_choice: 'Optional[ChatCompletionToolChoiceOption]' = None, temperature: 'float' = 0.2, top_p: 'float' = 0.95, top_k: 'int' = 40, min_p: 'float' = 0.05, typical_p: 'float' = 1.0, stream: 'bool' = False, stop: 'Optional[Union[str, List[str]]]' = [], seed: 'Optional[int]' = None, response_format: 'Optional[ChatCompletionRequestResponseFormat]' = None, max_tokens: 'Optional[int]' = None, presence_penalty: 'float' = 0.0, frequency_penalty: 'float' = 0.0, repeat_penalty: 'float' = 1.0, tfs_z: 'float' = 1.0, mirostat_mode: 'int' = 0, mirostat_tau: 'float' = 5.0, mirostat_eta: 'float' = 0.1, model: 'Optional[str]' = None, logits_processor: 'Optional[LogitsProcessorList]' = None, grammar: 'Optional[LlamaGrammar]' = None, logit_bias: 'Optional[Dict[int, float

In [19]:
print("Qwen3.5-9B story generator — type 'exit' to quit.\n")

while True:
    try:
        parent_input = input("Parent's description: ").strip()
    except (EOFError, KeyboardInterrupt):
        break

    if not parent_input or parent_input.lower() in {"exit", "quit"}:
        print("Ending session.")
        break

    print("\n---\n")
    generate_story(llm, parent_input)
    print("\n---\n")

Qwen3.5-9B story generator — type 'exit' to quit.

Parent's description: my child was chased by a dog today, now they are afraid of every dog that crosses his path, i need him to not be scared of animals like this

---

**Title: The Brave Little Cloud and the Big, Shaky Dog**

High above the town, a little cloud named Puff floated happily.
Puff loved to chase butterflies.
He loved to wiggle and dance in the sunshine.

One day, a big brown dog ran past below.
The dog was very fast!
"Whoa!" went Puff.
His tummy felt like it had a little drum beating inside. *Thump-thump.*
Puff froze in the sky.
He covered his eyes with his tiny arms.
"I don't like that dog," whispered Puff. "I am scared."

Down below, the dog was not being mean.
The dog just wanted to play tag with a ball.
But the dog ran so fast he looked like a blur.
To Puff, it looked like a storm coming.

Puff felt very small and very shaky.
He wished he could hide behind the moon.

Then, a wise old owl named Oliver flew by.
"Hello, 

## 7. Expose the story generator as an API (FastAPI + ngrok)

This turns the notebook into a small HTTP API so an outside app (e.g. a React Native app + your own backend) can call `POST /generate-story` and get a story back. It reuses the `llm` and `SYSTEM_PROMPT` already loaded above — the model is **not** reloaded.

### 7.1 Install the API requirements

In [20]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 102, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 106, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 478, in run
    requirement_set = resolver.resolve(
        reqs, check_supported_wheels=not options.target_dir
    )
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 103, in resolve
    result = self._result = resolver.resolve(
                            ~~~~~~~~~~~~~~~~^
        collected.requirements, max_rounds=limit_how_complex_resolution_can_be
        ^^^^^

### 7.2 ngrok auth token

ngrok requires a free account and auth token to open a tunnel. Get one at [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken), then paste it below (it's only stored for this Colab session).

In [22]:
!pip install -q pyngrok

In [24]:
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3IMBqHDHgs6aEjkwKzknMdspbSv_4yHDxjLpNCqwM5Zmd3DzL"  # paste your ngrok auth token here

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
else:
    print("No auth token set — set NGROK_AUTHTOKEN above before running the tunnel cell below.")

### 7.3 Define the API endpoint

Requires `llm` (step 4) and `SYSTEM_PROMPT` (step 5) to already be defined — run those cells first if you haven't.

In [ ]:
import random
from fastapi import FastAPI
from pydantic import BaseModel

assert "llm" in globals(), "llm is not defined — run step 4 (load the model) first."
assert "SYSTEM_PROMPT" in globals(), "SYSTEM_PROMPT is not defined — run step 5 first."

app = FastAPI()


class StoryRequest(BaseModel):
    description: str


@app.post("/generate-story")
async def generate_story_endpoint(request: StoryRequest):
    # Same chat prompt template as step 5, just filled in from the request body.
    prompt = build_user_message(request.description)

    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.85,
        top_p=0.9,
        top_k=40,
        repeat_penalty=1.05,
        presence_penalty=1.5,
        max_tokens=500,
        seed=random.randint(0, 2**32 - 1),
    )

    text = response["choices"][0]["message"]["content"]
    # Strip any leftover <think>...</think> block, same as the interactive loop.
    import re
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    return {"story": text}


@app.get("/health")
async def health():
    return {"status": "ok"}

### 7.4 Start the server in the background

Running `uvicorn.run(...)` directly would block the cell forever. Instead we run it in a background thread so the notebook stays usable — re-run the interactive loop in step 6, inspect `llm`, etc. while the API keeps serving requests.

In [26]:
import threading
import time

import nest_asyncio
import uvicorn

nest_asyncio.apply()


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)  # give uvicorn a moment to bind the port
print("Server thread started on port 8000.")

INFO:     Started server process [1606]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server thread started on port 8000.


### 7.5 Open the ngrok tunnel

This exposes local port 8000 publicly. Copy the printed URL — that's your API base URL, and your endpoint is `<that URL>/generate-story`.

In [27]:
# Close any tunnels left open from a previous run of this cell.
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(8000)
print("Public base URL:", public_url)
print("Story endpoint: ", str(public_url) + "/generate-story")
print("Health check:   ", str(public_url) + "/health")

Public base URL: NgrokTunnel: "https://studied-knoll-filler.ngrok-free.dev" -> "http://localhost:8000"
Story endpoint:  NgrokTunnel: "https://studied-knoll-filler.ngrok-free.dev" -> "http://localhost:8000"/generate-story
Health check:    NgrokTunnel: "https://studied-knoll-filler.ngrok-free.dev" -> "http://localhost:8000"/health


### 7.6 Quick test from inside the notebook

Sanity-check the endpoint before wiring up your app.

In [30]:
import requests

ngrok_url = str(public_url).split('"')[1]

print("Using URL:", ngrok_url)

test_resp = requests.post(
    f"{ngrok_url}/generate-story",
    json={
        "description": "Today Ali didn't want to go to school because his friends laughed when he answered a question wrong."
    },
    timeout=120,
)

print(test_resp.status_code)
print(test_resp.json())

Using URL: https://studied-knoll-filler.ngrok-free.dev
INFO:     34.16.160.77:0 - "POST /generate-story HTTP/1.1" 200 OK
200
{'story': '**Title: The Brave Question Mark**\n\nAli loved school. He loved the bright colors of the classroom and the smell of fresh crayons.\nToday, Ali walked into his class with a small smile.\nBut then, Mr. Bear asked a question about the sky.\n"Who knows how many clouds are in the blue sky?" Mr. Bear asked.\n\nAli\'s tummy did a little flip-flop. *Flip... flop.*\n"I know!" Ali said loudly. "There are just one!"\n\nEveryone looked up. The class was quiet for a moment.\nThen, two friends giggled. They laughed softly.\nAli felt his face turn hot. His cheeks felt like warm toast.\nHe wanted to hide under his desk. He wished he could disappear.\n"Maybe I am wrong," Ali thought. "Maybe clouds are many."\n\nMr. Bear smiled gently. He walked over to Ali\'s chair.\n"Thank you, Ali," Mr. Bear said softly. "That was a very brave guess!"\nAli looked at his friends. The

### 7.7 App architecture

```
React Native
     ↓
POST /generate-story
     ↓
Your FastAPI backend
     ↓
https://abc123.ngrok-free.app/generate-story
     ↓
Colab FastAPI
     ↓
Qwen 3.5 9B
     ↓
Story
```

In your local backend, point at the ngrok URL printed in 7.5, e.g. (PowerShell):

```powershell
$env:QWEN_API_URL="https://abc123.ngrok-free.app"
```

**Notes:**
- The free ngrok URL changes every time you rerun 7.5 (new Colab session, tunnel dropped, etc.) — update `QWEN_API_URL` in your backend whenever it changes.
- Keep this Colab tab open and the runtime connected; if Colab disconnects, the server and tunnel both go down and you'll need to re-run from step 1.
- This setup has no auth on `/generate-story` — anyone with the ngrok URL can call it. Fine for local testing; add an API key check in the endpoint (e.g. a header your backend sends and the endpoint verifies) before sharing the URL more widely.

## Notes & troubleshooting

- **Out of memory when loading**: lower `n_ctx` in step 4 (e.g. `4096`), or switch to a smaller quant in step 3 (`UD-Q3_K_XL` or `UD-Q2_K_XL`).
- **Slow generation**: confirm step 2's sanity check printed `True` for `llama_supports_gpu_offload()` — if it printed `False`, the build didn't pick up CUDA and you're running on CPU. Re-run step 2 after restarting the runtime.
- **Session disconnects**: Colab free tier has usage limits and will disconnect idle or long-running sessions; the model download and compiled wheel aren't preserved between sessions, so re-running from the top is normal.
- **Prompts are fixed on purpose**: the system prompt and chat prompt template in step 5 are kept exactly as specified, so if you want to change tone, age range, or output format, edit `SYSTEM_PROMPT` / `CHAT_PROMPT_TEMPLATE` directly there.
- **Want the model's reasoning visible?** Remove the `chat_template_kwargs={"enable_thinking": False}` line in step 5 and skip the `<think>` stripping if you'd rather see Qwen3.5's chain-of-thought (not recommended here, since only the Title/Story should be shown to a parent).
